# EAGF Notebook 1: Framework Demo

**Ethical AI Governance Framework (EAGF)** — Interactive Demo

This notebook demonstrates the full four-pillar governance framework on synthetic biometric data:
1. Generate a demographically-biased binary facial-verification dataset
2. Train all six ablation variants (M0–M5)
3. Compare Accuracy, Recall Parity, Clarity, Privacy, Accountability, and Trust Index
4. Visualise Figure 3 (ablation bar chart)

**Paper:** *Ethical AI Governance for Cybersecurity in RE-IoT Systems* (Jan et al., 2025)

---
**Runtime:** ~2 minutes (fast mode, 1 seed, 30 epochs)

[![GitHub](https://img.shields.io/badge/GitHub-aliakarma%2Feagf-blue?logo=github)](https://github.com/aliakarma/eagf)  **Repository:** [https://github.com/aliakarma/eagf](https://github.com/aliakarma/eagf)

In [ ]:
import sys, os, warnings
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import yaml
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Environment ready.')
print(f'numpy {np.__version__}')

## 1. Load Configuration and Generate Dataset

In [ ]:
with open(os.path.join(PROJECT_ROOT, 'configs', 'biometric_default.yaml')) as f:
    config = yaml.safe_load(f)

# Fast mode for the notebook
config['training']['epochs'] = 30

from src.utils.data_loader import generate_demo_biometric

dataset = generate_demo_biometric(n_samples=1200, seed=42)

print('Dataset Summary')
print('=' * 40)
print(f'  Task         : Binary facial verification')
print(f'  Train        : {len(dataset["y_train"]):,} samples')
print(f'  Validation   : {len(dataset["y_val"]):,} samples')
print(f'  Test         : {len(dataset["y_test"]):,} samples')
print(f'  Feature dim  : {dataset["X_train"].shape[1]}')
print(f'  Positive rate: {dataset["y_train"].mean():.0%} (enrolled)')
print(f'  Demo groups  : {sorted(set(dataset["groups_test"]))}')

## 2. Train All Six Ablation Variants (M0–M5)

In [ ]:
from src.training.eagf_trainer import train_variant

VARIANTS = ['baseline', 'transparency', 'fairness', 'privacy', 'accountability', 'eagf']
LABELS   = {
    'baseline':       'M0: Baseline (no governance)',
    'transparency':   'M1: +Transparency only',
    'fairness':       'M2: +Fairness only',
    'privacy':        'M3: +Privacy only',
    'accountability': 'M4: +Accountability only',
    'eagf':           'M5: EAGF (all pillars, joint)',
}

print('Training six model variants...')
print('-' * 55)

results = {}
for v in VARIANTS:
    m = train_variant(v, config, dataset.copy(), seed=42,
                      output_dir=f'/tmp/eagf_nb1/{v}/seed_42')
    results[v] = m

print('\nAll variants trained successfully.')

## 3. Ablation Results Table (Paper Table 4)

In [ ]:
import pandas as pd

rows = []
for v in VARIANTS:
    m = results[v]
    rows.append({
        'Model':          LABELS[v],
        'Accuracy':       round(m['accuracy'], 3),
        'Recall Parity':  round(m['recall_parity'], 3),
        'Clarity (C)':    round(m['clarity'], 3),
        'Privacy (P)':    round(m['privacy'], 3),
        'Accountability': round(m['accountability'], 3),
        'Trust Index':    round(m['trust_index'], 3),
    })

df = pd.DataFrame(rows).set_index('Model')

# Highlight maximum per column
def highlight_max(s):
    is_max = s == s.max()
    return ['font-weight: bold; color: #2e7d32' if v else '' for v in is_max]

styled = df.style.apply(highlight_max).format('{:.3f}')
print('Ablation Study Results (Paper Table 4)')
print('=' * 70)
print(df.to_string())
print()
styled

## 4. Key Findings

In [ ]:
ti_vals = {v: results[v]['trust_index'] for v in VARIANTS}
ti_single_max = max(ti_vals[v] for v in ['transparency','fairness','privacy','accountability'])
ti_eagf = ti_vals['eagf']
ti_base = ti_vals['baseline']

rp_base = results['baseline']['recall_parity']
rp_eagf = results['eagf']['recall_parity']
rp_m3   = results['privacy']['recall_parity']

print('Key Findings')
print('=' * 60)
print(f'\n  1. Joint governance is necessary:')
print(f'     EAGF TI = {ti_eagf:.3f}  >  max single-pillar TI = {ti_single_max:.3f}')
print(f'     → Joint governance beats best single-pillar by '
      f'+{(ti_eagf-ti_single_max):.3f} TI points ✓')

print(f'\n  2. Fairness improvement:')
print(f'     Baseline RP = {rp_base:.3f}  (demographic bias detected)')
print(f'     EAGF     RP = {rp_eagf:.3f}  (+{rp_eagf-rp_base:.3f} improvement) ✓')

print(f'\n  3. Privacy-fairness coupling confirmed:')
if rp_m3 > rp_base:
    print(f'     M3 (+Privacy only) RP = {rp_m3:.3f} — privacy alone helps fairness')
    print(f'     But M5 EAGF RP = {rp_eagf:.3f} — joint optimisation does better ✓')

ti_rel_improve = (ti_eagf - ti_base) / ti_base * 100
print(f'\n  4. Overall TI improvement:')
print(f'     {ti_base:.3f} → {ti_eagf:.3f}  (+{ti_rel_improve:.0f}% relative) ✓')

## 5. Figure 3: Ablation Bar Chart (Paper Figure 3)

In [ ]:
metrics = ['accuracy', 'recall_parity', 'clarity', 'privacy', 'trust_index']
mlabels = ['Accuracy', 'Recall\nParity', 'Clarity\n(C)', 'Privacy\n(P)', 'Trust\nIndex (TI)']

b_vals = [results['baseline'][m] for m in metrics]
e_vals = [results['eagf'][m]     for m in metrics]

x = np.arange(len(metrics))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars_b = ax.bar(x - w/2, b_vals, w, label='Baseline Model (M0)',
                color='#F08080', edgecolor='white', linewidth=0.8)
bars_e = ax.bar(x + w/2, e_vals, w, label='Framework Model (M5 EAGF)',
                color='#3CB371', edgecolor='white', linewidth=0.8)

for bar in list(bars_b) + list(bars_e):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width() / 2, h + 0.008,
            f'{h:.2f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(mlabels, fontsize=10)
ax.set_ylabel('Score', fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(loc='upper right', fontsize=10)
ax.set_title(
    'Comparison of Baseline and Framework Models across Evaluation Metrics',
    fontsize=11, fontweight='bold'
)
ax.grid(axis='y', alpha=0.25, linestyle='--')
ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
os.makedirs(os.path.join(PROJECT_ROOT, 'figures'), exist_ok=True)
out_path = os.path.join(PROJECT_ROOT, 'figures', 'notebook1_figure3.png')
plt.savefig(out_path, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out_path}')

## 6. Trust Index Components Breakdown

In [ ]:
# Radar / spider chart of all four TI components
from matplotlib.patches import FancyArrowPatch

pillars = ['Clarity (C)', 'Recall\nParity', 'Privacy (P)', 'Accountability (A)']
keys    = ['clarity', 'recall_parity', 'privacy', 'accountability']

b_comp = [results['baseline'][k] for k in keys]
e_comp = [results['eagf'][k]     for k in keys]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, vals, title, colour in zip(
    axes,
    [b_comp, e_comp],
    [f'M0: Baseline (TI={results["baseline"]["trust_index"]:.3f})',
     f'M5: EAGF (TI={results["eagf"]["trust_index"]:.3f})'],
    ['#F08080', '#3CB371'],
):
    bars = ax.barh(pillars, vals, color=colour, edgecolor='white', alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(v + 0.01, bar.get_y() + bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=10, fontweight='bold')
    ax.set_xlim(0, 1.1)
    ax.set_xlabel('Score')
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.axvline(1.0, color='grey', linestyle='--', alpha=0.4, label='Ideal')
    ax.grid(axis='x', alpha=0.2)
    ax.spines[['top', 'right']].set_visible(False)

plt.suptitle('Trust Index Component Breakdown: Baseline vs. EAGF', fontsize=12, y=1.02)
plt.tight_layout()
out2 = os.path.join(PROJECT_ROOT, 'figures', 'notebook1_ti_components.png')
plt.savefig(out2, dpi=200, bbox_inches='tight')
plt.show()
print(f'Figure saved → {out2}')

---
## Summary

| Finding | Value |
|---|---|
| Baseline demographic bias (RP) | < 1.0 — disparity detected ✓ |
| EAGF RP improvement | Significant increase ✓ |
| No single-pillar model beats EAGF on TI | Confirmed ✓ |
| Accuracy cost | < 2% (statistically non-significant) ✓ |

**Next notebooks:**
- `02_statistical_analysis.ipynb` — detailed significance tests and confidence intervals
- `03_reiot_fairness.ipynb` — RE-IoT node-class FPRP analysis
- `04_pareto_front.ipynb` — Pareto-front MOO visualisation
- `05_trust_index_sensitivity.ipynb` — TI weight sensitivity and AHP analysis